## Подключение

Ячейка подключения выполняется первой в каждом сеансе. Она подключает Google Drive (личная папка `practice2026` и общие данные группы `kg_data`), клонирует репозиторий группы в `/content/kg-practice-2026`, переключается на свою ветку `student-NN` с работой группы из `main` и регистрирует модули группы. В первых строках студент указывает свой номер, модуль и фамилию латиницей; токен GitHub лежит в секретах Colab под именем `GITHUB_TOKEN` (`ИНСТРУКЦИЯ_git.md`). Сдача этапа — последняя ячейка блокнота: `repo.submit(...)`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import sys, os, json, time, subprocess

STUDENT = 'student-07'                     # ← свой номер: вариант 7 — student-07
MY_MODULE = 'm07'                          # ← свой модуль: вариант 7 — m07
SURNAME = 'ivanov'                         # ← фамилия латиницей строчными буквами
REPO = 'github.com/<организация>/kg-practice-2026'     # репозиторий группы

KG = '/content/kg-practice-2026'                       # клон репозитория: код, модули, эталоны, сдача
DATA = '/content/drive/MyDrive/kg_data'                # общие данные группы на Drive (только чтение)
WORK = '/content/drive/MyDrive/practice2026'           # личная папка на Drive: свой набор и результаты
TEST = DATA + '/test_set'                              # общий тестовый набор
TEST_FACTS = WORK + '/test_facts'                      # свои прогоны модулей на тестовом наборе
ONTOLOGY = [KG + '/ontology/MocassinPlus_v2.ttl', KG + '/ontology/SciLibReco.ttl']
os.makedirs(WORK, exist_ok=True)

TOKEN = userdata.get('GITHUB_TOKEN')                   # секрет Colab с токеном GitHub
if not os.path.exists(KG):
    r = subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{REPO}.git', KG], capture_output=True, text=True)
    print(r.stderr.replace(TOKEN, '***'))
sys.path.insert(0, KG)
!pip install -q -r {KG}/requirements.txt
from mathkg.gitwork import Repo
repo = Repo(KG, STUDENT, MY_MODULE, SURNAME, token=TOKEN)
repo.start()                                           # своя ветка и всё, что уже принято в main
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
print('свой модуль:', MY_MODULE, my_tool)

# Этап 03. Модуль: правила и ручной эталон

Лекция 3 · сдать к встрече 5 (сборка 1) · работа студента: 7–8 ч, из них разметка эталона 1,5–2 ч ·
вычисления в Colab: прогон модуля на тестовом наборе — секунды; оценка — секунды.

## Цель

Студент доводит правила своего модуля до рабочего состояния и впервые измеряет качество: размечает
вручную объекты своего модуля на 10 статьях общего тестового набора, считает точность, полноту и F1
базовой версии и своей. Правила опираются на то, что даёт base: заголовки блоков, отметки шрифта,
положение на странице, гиперссылки PDF. К встрече 5 модуль проходит проверку контракта и попадает в
сборку 1.

## Что прочитать

* SLP3, гл. 18 «Sequence Labeling for Parts of Speech and Named Entities»: схема BIO, оценка
  разметки на уровне сущностей.
* Alammar, Grootendorst, гл. 11, разд. «Named-Entity Recognition».
* `02_МОДЕЛЬ_ДАННЫХ.md` §2 (локальные идентификаторы), §5 (формат результата модуля, формат
  эталона), §6 (что даёт базовая версия и что добавляет студент).
* Исходный код: `mathkg/model.py`, `mathkg/evaluate.py`, `mathkg/check.py`.

## Вход

* `modules/mNN_*.py` — модуль после этапа 2.
* Тестовый набор: `kg_data/test_set/base/`; результаты модулей — `practice2026/test_facts/`.
* Список 10 статей эталона своего модуля (выдаёт руководитель, чтобы эталоны разных модулей
  покрывали разные журналы).

## Шаги

### 1. Признаки, которые даёт base

Правила модуля строятся на четырёх источниках сведений о статье:

| Источник | Поле `Doc` | Пример использования |
|---|---|---|
| текст | `text`, `paragraphs`, `sentences` | заголовок блока «Теорема 2.1.» в начале абзаца |
| шрифт | `styled` (`bold`, `italic`, `math`) | заголовок блока полужирный; формулировка теоремы и вводимый термин — курсив |
| положение | `first_page_lines` (кегль, `bbox`), `formulas[i].bbox` | заголовок статьи — самый крупный кегль верхней половины первой страницы |
| гиперссылки PDF | `links` (`target`, `start`, `end`) | ссылка с целью `cite.Diestel75` указывает на запись литературы; `equation.2.3` — на формулу (2.3) |

Помощники для правил:

In [ ]:
import pandas as pd
from mathkg.base import read_doc
from mathkg import registry, pipeline
from mathkg.modules._util import italic_share, is_bold, header_re, paragraph_at, NUM
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
test_ids = pd.read_json(TEST + '/base/docs.jsonl', lines=True).query("status == 'ok'").doc_id.tolist()
docs = {i: read_doc(TEST + '/base', i) for i in test_ids}

d = docs[test_ids[0]]
p = d.paragraphs[12]
print(d.text[p.start:p.end][:100])
print('курсив:', round(italic_share(d, p.start, p.end), 2), '| полужирный:', is_bold(d, p.start, p.end))
print('отметки:', d.style_at(p.start, p.start + 20))
print('гиперссылки абзаца:', [l.target for l in d.links if p.start <= l.start < p.end])

Гиперссылки дают точные связи для статей, свёрстанных LaTeX, и пусты для статей из Word. Поэтому
правило строится так, чтобы ссылка повышала уверенность, а её отсутствие не отключало правило.

### 2. Разметка последовательностей и место NER

Лекция 3 рассматривает разметку последовательностей: каждому токену ставится метка (BIO), модель
NER находит имена, организации, места. Для трёх модулей готовая модель NER полезна уже на этапе
правил как проверка:

* m01 (метаданные) — имена авторов в строках под заголовком;
* m06 (литература) — фамилии авторов в записях списка литературы;
* m09 (понятия) — термины, которые модель отмечает как сущности (слабая подсказка: модели обучены
  на персонах и организациях).

In [ ]:
from transformers import pipeline as hf_pipeline
ner = hf_pipeline('token-classification', model='Babelscape/wikineural-multilingual-ner',
                  aggregation_strategy='simple', device=0)
line = ' '.join(l['text'] for l in d.first_page_lines[1:6])
print([(e['word'], e['entity_group'], round(e['score'], 2)) for e in ner(line)])

Для русского текста дополнительно подходит Natasha (`pip install natasha`, модель `slovnet`),
работающая на CPU. Остальные модули на этом этапе обходятся правилами.

### 3. Правила модуля

Студент дорабатывает функцию модуля в `modules/mNN_*.py`. Требования формата
(проверяет `check_module`):

* фрагмент — `facts.add_fragment(id=..., cls=..., start=..., end=..., number=..., label=...,
  page=..., method='rule', confidence=...)`; `id` — с префиксом своего модуля (§2 модели данных),
  без `:` и `/`;
* связь — `facts.add(subj, pred, obj, confidence=..., start=..., end=...)`; `subj` и `obj` —
  локальный id своего фрагмента, фрагмента модуля-зависимости или base (`p-0012`, `f-0003`), либо
  полный IRI;
* класс и свойство — из онтологий;
* `confidence` отражает надёжность правила: 0,9 для правила с подтверждением шрифтом или
  гиперссылкой, 0,6–0,7 для правила только по тексту.

Перезагрузка и пересчёт после каждой правки:

In [ ]:
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
pipeline.run_corpus(TEST + '/base', TEST_FACTS, redo=[my_tool])

### 4. Ручной эталон

Эталон — единственный способ измерить качество модуля: исходников статей нет. Студент размечает
объекты своего модуля на 10 статьях из выданного списка и записывает их в `gold/mNN.jsonl`
по строке на объект. Позиции вручную не считаются: в эталон записывается дословная цитата начала
фрагмента длиной 30–80 символов, скопированная из `Doc.text`.

```json
{"doc_id": "A04571", "cls": "moc:Theorem", "number": "2.1", "quote": "Теорема 2.1. Пусть X — рефлексивное"}
{"doc_id": "A04571", "pred": "moc:proves", "subj_quote": "Доказательство. По лемме 1.3", "obj_quote": "Теорема 2.1. Пусть X"}
```

Порядок разметки:

1. студент открывает PDF статьи и `Doc.text` рядом, идёт по статье от начала до конца;
2. каждый объект своего модуля записывает строкой: класс, номер (если есть), цитата;
3. связи своего модуля (если модуль их создаёт) записывает с цитатами субъекта и объекта;
4. спорные случаи записывает в отчёт с решением, которое принял;
5. проверяет, что все цитаты находятся в тексте:

In [ ]:
from mathkg.evaluate import read_gold, locate
gold = read_gold(f'{KG}/gold/{MY_MODULE}.jsonl')
for g in gold:
    q = g.get('quote') or g.get('subj_quote')
    if locate(read_doc(TEST + '/base', g['doc_id']), q) < 0:
        print('не найдено:', g)

Эталон размечается по тексту статьи, без оглядки на то, что находит модуль: иначе оценка будет
завышена. Разметка 10 статей занимает 1,5–2 часа.

### 5. Оценка

`evaluate` сравнивает результат модуля с эталоном: фрагмент найден, если модуль выдал фрагмент того
же класса с началом не дальше 15 символов от цитаты; связь найдена, если модуль выдал связь с тем же
свойством между фрагментами, содержащими цитаты.

In [ ]:
from mathkg.evaluate import evaluate
import pandas as pd

def score(tag):
    r = evaluate(my_tool, f'{KG}/gold/{MY_MODULE}.jsonl', TEST + '/base', TEST_FACTS)
    df = pd.DataFrame(r['by_key']).T.assign(version=tag)
    return df, r

# базовая версия: пересчёт без студенческих модулей
pipeline.setup(); pipeline.run_corpus(TEST + '/base', TEST_FACTS, redo=[my_tool])
base_df, _ = score('базовая')
pipeline.setup(student_modules=KG + '/modules'); pipeline.run_corpus(TEST + '/base', TEST_FACTS, redo=[my_tool])
mine_df, r = score('правила')
print(pd.concat([base_df, mine_df]))
print('пропуски:', r['missed'][:10]); print('ложные:', r['extra'][:10])

Вызов `pipeline.setup()` без папки модулей сбрасывает реестр и возвращает базовые версии всех
модулей, поэтому обе версии сравниваются в одном сеансе.

### 6. Проверка контракта и сборка 1

In [ ]:
from mathkg.check import check_module
rep = check_module(my_tool, TEST + '/base', TEST_FACTS, ONTOLOGY)
print(rep['passed'], rep['ms_per_doc'], rep['errors'][:10])

Модуль сдаётся к встрече 5, если `passed` — `True`. На встрече руководитель собирает модули группы
(`build_report`) и показывает таблицу сборки 1. Модуль, не прошедший проверку, остаётся в сборке в
базовой версии.

## Выход

* `modules/mNN_*.py` — модуль на правилах;
* `gold/mNN.jsonl` — эталон на 10 статьях;
* `practice2026/stage03/REPORT.md` (ячейка сдачи копирует его в `students/NN_фамилия/stage03/`).

## Самопроверка

* Все цитаты эталона находятся в тексте (`locate` ≥ 0).
* F1 своей версии на эталоне выше, чем у базовой, хотя бы по одному классу; если ниже — в отчёте
  разобраны причины.
* `check_module` проходит без ошибок, среднее время — миллисекунды или десятки миллисекунд.
* Списки пропусков и ложных находок просмотрены; для каждой группы ошибок есть объяснение.

## Что сдать

Сдача — ячейка сдачи в конце блокнота и запрос на слияние «Этап 3: фамилия» в `main`
(`ИНСТРУКЦИЯ_git.md`, раздел 4). В репозиторий попадают модуль `modules/mNN_*.py` и эталон `gold/mNN.jsonl`, блокнот
`students/NN_фамилия/stage03.ipynb` с выводами и папка `students/NN_фамилия/stage03/` с отчётом
`REPORT.md` и небольшими результатами этапа. В отчёте: описание
правил и признаков (текст, шрифт, положение, гиперссылки), таблица P/R/F1 базовой версии и своей по
классам и связям, разбор 10 ошибок, спорные случаи разметки.

## Как результат станет инструментом агента

Модуль уже является инструментом: `registry.agent_tools` строит из него функцию
`<имя>(doc_id) -> {fragments, relations}` с описанием из docstring. Студент переписывает docstring
своей функции так, чтобы по нему модель агента поняла, когда вызывать инструмент и что он вернёт.

## Частые ошибки

* Эталон размечен по результату модуля, и оценка завышена.
* Цитата скопирована из PDF: в ней лигатуры или переносы, и `locate` её не находит. Цитаты
  копируются из `Doc.text`.
* Порог уверенности одинаков для всех правил; агент и этап 6 не смогут отличить надёжные находки от
  сомнительных.
* Фрагменты разных классов получают одинаковые `id`.
* Правило работает только при наличии гиперссылок и молчит на статьях из Word.

## Вопросы к защите

1. Что такое схема BIO и как по ней считаются точность и полнота на уровне сущностей?
2. Почему частичное совпадение границ оценивается отдельно от точного?
3. Какие признаки base использует ваше лучшее правило и почему оно надёжно?
4. Чем точность отличается от полноты на примере вашего модуля, и что важнее для следующих модулей?
5. Как гиперссылки PDF меняют уверенность связи и почему правило не должно от них зависеть целиком?
6. Почему эталон размечается без оглядки на результат модуля?
7. Где в вашем модуле уместна модель NER и что она добавит к правилам?

**Вариант.** Какие объекты размечать в эталоне, как поступать в спорных случаях и какие правила
дают наибольший прирост для своего модуля, описано в разделе «Этап 3» файла
`варианты/Вариант_NN.md`.

## Сдача этапа

Перед сдачей блокнот сохраняют (Ctrl+S), а отчёт `REPORT.md` кладут в папку этапа на Drive (`practice2026/stageNN/`). Ячейка копирует в свою папку сдачи `students/NN_фамилия/` блокнот и небольшие файлы этапа (отчёт, таблицы, запросы, журналы), отправляет свои файлы в ветку `student-NN` и печатает ссылку на запрос на слияние в `main`; запрос открывают по ссылке с заголовком «Этап N: фамилия».

In [ ]:
STAGE = 3
repo.save_notebook(f'stage{STAGE:02d}_{SURNAME}.ipynb', STAGE)   # имя своей копии блокнота на Drive
repo.collect(f'{WORK}/stage{STAGE:02d}', STAGE)                  # отчёт и небольшие результаты этапа
repo.submit(f'Этап {STAGE}: {SURNAME}')